In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [4]:
DATA_FILE = Path(
    r"C:\Users\admin\Desktop\HWU\ResearchMethods\MachineLearningCVE\processed\cicids2017_clean.csv"
)

data = pd.read_csv(DATA_FILE)

print("Dataset shape:", data.shape)
print(data["BinaryLabel"].value_counts())

Dataset shape: (2520798, 80)
BinaryLabel
0    2095057
1     425741
Name: count, dtype: int64


In [5]:
X = data.drop(
    columns=[
        "Label",
        "BinaryLabel",
        "SourceFile"
    ]
)

y = data["BinaryLabel"]

print("Number of ML features:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of ML features: 77
X shape: (2520798, 77)
y shape: (2520798,)


In [6]:
pilot_data = data.sample(
    n=300_000,
    random_state=42
)

X_pilot = pilot_data.drop(
    columns=[
        "Label",
        "BinaryLabel",
        "SourceFile"
    ]
)

y_pilot = pilot_data["BinaryLabel"]

print("Pilot dataset:", X_pilot.shape)

print("\nPilot class distribution:")
print(y_pilot.value_counts(normalize=True).mul(100))

Pilot dataset: (300000, 77)

Pilot class distribution:
BinaryLabel
0    83.031667
1    16.968333
Name: proportion, dtype: float64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pilot,
    y_pilot,
    test_size=0.20,
    random_state=42,
    stratify=y_pilot
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).mul(100))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).mul(100))

Training samples: 240000
Test samples: 60000

Training class distribution:
BinaryLabel
0    83.031667
1    16.968333
Name: proportion, dtype: float64

Test class distribution:
BinaryLabel
0    83.031667
1    16.968333
Name: proportion, dtype: float64


In [8]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print(rf)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)


In [9]:
rf.fit(X_train, y_train)

print("Training completed.")

Training completed.


In [10]:
y_pred = rf.predict(X_test)

print("Predictions completed.")

Predictions completed.


In [11]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()

false_positive_rate = fp / (fp + tn)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall / Detection Rate: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"False Positive Rate: {false_positive_rate:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy: 0.9982
Precision: 0.9959
Recall / Detection Rate: 0.9935
F1-score: 0.9947
False Positive Rate: 0.0008

Confusion Matrix:
[[49777    42]
 [   66 10115]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     49819
      ATTACK       1.00      0.99      0.99     10181

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000

